# Notebook 11 — Minor Revision Plot Evidence

This notebook contains minor and editorial figure revisions. Shared imports, paths, and export helpers are defined once. Each reviewer concern is then added as a compact, separately labelled section containing: figure construction, in-notebook preview, and optional export.

In [ ]:
# ==========================================================
# Shared setup — imports, project paths, and export helper
# ==========================================================

from pathlib import Path
import gc
import os

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Image, Markdown, display

DEFAULT_ROOT = Path(
    "/Users/reneprincipejr/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)/"
    "blackmarble-reliability-conditions"
)

cwd_root = Path.cwd()
if cwd_root.name == "notebooks":
    cwd_root = cwd_root.parent

PROJECT_ROOT = cwd_root if (cwd_root / "datasets").exists() else Path(
    os.environ.get("RQ1_PROJECT_ROOT", DEFAULT_ROOT)
).expanduser()

DATASET_DIR = PROJECT_ROOT / "datasets"
OUTPUT_MAIN_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "main"
OUTPUT_SUPP_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "supplementary"

for output_dir in [OUTPUT_MAIN_FIG_DIR, OUTPUT_SUPP_TABLE_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)


def export_plotly_figure(fig, output_path, width, height, scale=2):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig.write_image(
        str(output_path.with_suffix(".png")),
        width=width,
        height=height,
        scale=scale,
    )
    fig.write_image(
        str(output_path.with_suffix(".svg")),
        width=width,
        height=height,
        scale=1,
    )

    print(f"Exported: {output_path.with_suffix('.png')}")
    print(f"Exported: {output_path.with_suffix('.svg')}")


print(f"Project root: {PROJECT_ROOT}")
print(f"Main figure output: {OUTPUT_MAIN_FIG_DIR}")

## R1.4 — Figure 3 NTL contrast

> “The NTL visualization in Figure 3 should use a more appropriate contrast stretch, as the current display makes spatial variations in NTL radiance difficult to distinguish.”

Four individual two-panel figures are prepared for 1, 5, 10, and 13 November 2013. A common 0–P99 linear display range is used across all DNB-BRDF panels. Statistics use the original radiance values. Regional boundaries are overlaid from `Regions.shp`, and Overleaf supplies the final subcaptions and multi-image layout.

In [ ]:
# ==========================================================
# R1.4 — Load inputs and define the Figure 3 builder
# ==========================================================

R14_TIFF_DIR = DATASET_DIR / "vnp46a2" / "region8_allbands_nov2013"
R14_REGIONS_PATH = DATASET_DIR / "boundaries" / "regions" / "Regions.shp"

R14_DATES = {
    "20131101": {"panel": "a", "label": "1 Nov 2013"},
    "20131105": {"panel": "b", "label": "5 Nov 2013"},
    "20131110": {"panel": "c", "label": "10 Nov 2013"},
    "20131113": {"panel": "d", "label": "13 Nov 2013"},
}

R14_FIG_WIDTH = 820
R14_FIG_HEIGHT = 500
R14_DISPLAY_PERCENTILE = 99

# Rasterio band numbers are one-based.
R14_BAND_DNB = 1
R14_BAND_GAP = 3
R14_BAND_MQF = 5

R14_QF_LABELS = [
    "Persistent (0)",
    "Ephemeral (1)",
    "Poor/Cloud (2)",
    "Fill (255)",
]

R14_QF_COLORSCALE = [
    [0.0000, "green"], [0.2499, "green"],
    [0.2500, "royalblue"], [0.4999, "royalblue"],
    [0.5000, "orange"], [0.7499, "orange"],
    [0.7500, "lightgrey"], [1.0000, "lightgrey"],
]

for required_path in [R14_TIFF_DIR, R14_REGIONS_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)


def r14_find_tiff(date_key):
    matches = sorted(R14_TIFF_DIR.glob(f"*{date_key}*.tif"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one GeoTIFF for {date_key}; found {len(matches)}: {matches}"
        )
    return matches[0]


def r14_read_band(src, band_number):
    return src.read(band_number, masked=True).filled(np.nan).astype(np.float32)


def r14_append_boundary(geometry, x_values, y_values):
    if geometry is None or geometry.is_empty:
        return

    if geometry.geom_type == "LineString":
        x_coords, y_coords = geometry.xy
        x_values.extend(x_coords)
        y_values.extend(y_coords)
        x_values.append(None)
        y_values.append(None)
        return

    if geometry.geom_type in {"Polygon", "MultiPolygon"}:
        r14_append_boundary(geometry.boundary, x_values, y_values)
        return

    if hasattr(geometry, "geoms"):
        for part in geometry.geoms:
            r14_append_boundary(part, x_values, y_values)


R14_TIFF_BY_DATE = {
    date_key: r14_find_tiff(date_key)
    for date_key in R14_DATES
}

with rasterio.open(R14_TIFF_BY_DATE["20131101"]) as src:
    left, bottom, right, top = src.bounds

R14_LON_RANGE = [float(left), float(right)]
R14_LAT_RANGE = [float(bottom), float(top)]

regions = gpd.read_file(R14_REGIONS_PATH).to_crs(epsg=4326)
visible_regions = regions.cx[left:right, bottom:top]
if visible_regions.empty:
    raise ValueError("Regions.shp does not intersect the raster extent")

R14_BOUNDARY_X = []
R14_BOUNDARY_Y = []
for geometry in visible_regions.geometry:
    r14_append_boundary(geometry, R14_BOUNDARY_X, R14_BOUNDARY_Y)

del regions, visible_regions

contrast_values = []
for path in R14_TIFF_BY_DATE.values():
    with rasterio.open(path) as src:
        ntl = r14_read_band(src, R14_BAND_DNB)
        gap = r14_read_band(src, R14_BAND_GAP)
    valid = np.isfinite(ntl) & np.isfinite(gap)
    contrast_values.append(ntl[valid])
    del ntl, gap, valid

contrast_values = np.concatenate(contrast_values)
R14_DISPLAY_MIN = 0.0
R14_DISPLAY_MAX = float(
    np.nanpercentile(contrast_values, R14_DISPLAY_PERCENTILE)
)
if not np.isfinite(R14_DISPLAY_MAX) or R14_DISPLAY_MAX <= 0:
    R14_DISPLAY_MAX = 1.0

R14_NTL_TICKS = np.linspace(R14_DISPLAY_MIN, R14_DISPLAY_MAX, 6)
del contrast_values
gc.collect()


def build_r14_figure(date_key):
    path = R14_TIFF_BY_DATE[date_key]

    with rasterio.open(path) as src:
        ntl = np.flipud(r14_read_band(src, R14_BAND_DNB))
        gap = np.flipud(r14_read_band(src, R14_BAND_GAP))
        mqf = np.flipud(r14_read_band(src, R14_BAND_MQF))
        left, bottom, right, top = src.bounds
        lons = np.linspace(left, right, src.width, dtype=np.float32)
        lats = np.linspace(bottom, top, src.height, dtype=np.float32)

    land_mask = np.isfinite(gap)
    valid_ntl = land_mask & np.isfinite(ntl)

    total_land_px = int(land_mask.sum())
    valid_ntl_px = int(valid_ntl.sum())
    spatial_completeness = (
        100 * valid_ntl_px / total_land_px if total_land_px else np.nan
    )

    stats = {
        "date": pd.to_datetime(date_key, format="%Y%m%d"),
        "valid_pixels": valid_ntl_px,
        "total_land_pixels": total_land_px,
        "spatial_completeness_pct": spatial_completeness,
        "dnb_brdf_mean": float(np.nanmean(ntl[valid_ntl])),
        "dnb_brdf_median": float(np.nanmedian(ntl[valid_ntl])),
        "gap_filled_mean": float(np.nanmean(gap[land_mask])),
        "gap_filled_median": float(np.nanmedian(gap[land_mask])),
    }

    mqf_plot = np.full(mqf.shape, np.nan, dtype=np.float32)
    mqf_land = mqf[land_mask].copy()
    mqf_land[~np.isfinite(mqf_land)] = 255
    mqf_land[mqf_land == 255] = 3
    mqf_plot[land_mask] = mqf_land
    del mqf_land

    fig = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=True,
        horizontal_spacing=0.15,
    )

    fig.add_trace(
        go.Heatmap(
            z=ntl,
            x=lons,
            y=lats,
            zmin=R14_DISPLAY_MIN,
            zmax=R14_DISPLAY_MAX,
            colorscale="Cividis",
            colorbar=dict(
                title=dict(text="nW/cm²/sr", side="top", font=dict(size=12)),
                tickvals=R14_NTL_TICKS,
                ticktext=[f"{value:.1f}" for value in R14_NTL_TICKS],
                tickfont=dict(size=10),
                len=0.50,
                thickness=13,
                x=0.475,
                y=0.66,
            ),
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Heatmap(
            z=mqf_plot,
            x=lons,
            y=lats,
            zmin=-0.5,
            zmax=3.5,
            colorscale=R14_QF_COLORSCALE,
            colorbar=dict(
                title=dict(text="Quality Flag", side="top", font=dict(size=12)),
                tickvals=[0, 1, 2, 3],
                ticktext=R14_QF_LABELS,
                tickfont=dict(size=10),
                len=0.50,
                thickness=13,
                x=1.025,
                y=0.66,
            ),
            hoverinfo="skip",
        ),
        row=1,
        col=2,
    )

    for column in [1, 2]:
        fig.add_trace(
            go.Scatter(
                x=R14_BOUNDARY_X,
                y=R14_BOUNDARY_Y,
                mode="lines",
                line=dict(color="rgba(35,35,35,0.90)", width=1.1),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

    for title, xref, yref in [
        ("DNB_BRDF_Corrected_NTL", "x domain", "y domain"),
        ("Mandatory_Quality_Flag", "x2 domain", "y2 domain"),
    ]:
        fig.add_annotation(
            text=title,
            xref=xref,
            yref=yref,
            x=0,
            y=1.035,
            xanchor="left",
            yanchor="bottom",
            showarrow=False,
            font=dict(family="Arial", size=13, color="black"),
        )

    fig.add_annotation(
        text=(
            f"DNB-BRDF — Mean: {stats['dnb_brdf_mean']:.2f}; "
            f"Median: {stats['dnb_brdf_median']:.2f}<br>"
            f"Gap-filled — Mean: {stats['gap_filled_mean']:.2f}; "
            f"Median: {stats['gap_filled_median']:.2f}"
        ),
        xref="x domain",
        yref="paper",
        x=0.5,
        y=-0.14,
        xanchor="center",
        yanchor="top",
        showarrow=False,
        align="center",
        font=dict(family="Arial", size=10, color="black"),
    )
    fig.add_annotation(
        text=f"Spatial Completeness: {spatial_completeness:.1f}%",
        xref="x2 domain",
        yref="paper",
        x=0.5,
        y=-0.14,
        xanchor="center",
        yanchor="top",
        showarrow=False,
        font=dict(family="Arial", size=10, color="black"),
    )

    fig.update_layout(
        template="plotly_white",
        width=R14_FIG_WIDTH,
        height=R14_FIG_HEIGHT,
        margin=dict(l=50, r=175, t=55, b=105),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="#E5ECF6",
        font=dict(family="Arial", size=11, color="black"),
    )
    fig.update_xaxes(
        range=R14_LON_RANGE,
        tickformat=".1f",
        tickfont=dict(size=9),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.70)",
        zeroline=False,
    )
    fig.update_yaxes(
        range=R14_LAT_RANGE,
        tickformat=".1f",
        tickfont=dict(size=9),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.70)",
        zeroline=False,
    )
    fig.update_yaxes(showticklabels=False, row=1, col=2)

    del ntl, gap, mqf, mqf_plot, land_mask, valid_ntl
    return fig, stats


print(f"Shared display range: 0–{R14_DISPLAY_MAX:.2f} nW/cm²/sr")
print(f"Regional boundary coordinates: {len(R14_BOUNDARY_X):,}")

In [ ]:
# ==========================================================
# R1.4 — Figure 3 builder and in-notebook preview
# ==========================================================

def build_r14_figure(date_key):

    path = R14_TIFF_BY_DATE[date_key]

    with rasterio.open(path) as src:

        ntl = np.flipud(
            r14_read_band(src, R14_BAND_DNB)
        )

        gap = np.flipud(
            r14_read_band(src, R14_BAND_GAP)
        )

        mqf = np.flipud(
            r14_read_band(src, R14_BAND_MQF)
        )

        left, bottom, right, top = src.bounds

        lons = np.linspace(
            left,
            right,
            src.width,
            dtype=np.float32,
        )

        lats = np.linspace(
            bottom,
            top,
            src.height,
            dtype=np.float32,
        )

    # ------------------------------
    # Statistics
    # ------------------------------

    land_mask = np.isfinite(gap)
    valid_ntl_mask = land_mask & np.isfinite(ntl)

    total_land_px = int(land_mask.sum())
    valid_ntl_px = int(valid_ntl_mask.sum())

    spatial_completeness = (
        100 * valid_ntl_px / total_land_px
        if total_land_px
        else np.nan
    )

    mean_ntl = float(
        np.nanmean(ntl[valid_ntl_mask])
    )

    median_ntl = float(
        np.nanmedian(ntl[valid_ntl_mask])
    )

    mean_gap = float(
        np.nanmean(gap[land_mask])
    )

    median_gap = float(
        np.nanmedian(gap[land_mask])
    )

    statistics = {
        "date": pd.to_datetime(
            date_key,
            format="%Y%m%d",
        ),
        "valid_pixels": valid_ntl_px,
        "total_land_pixels": total_land_px,
        "spatial_completeness_pct": spatial_completeness,
        "dnb_brdf_mean": mean_ntl,
        "dnb_brdf_median": median_ntl,
        "gap_filled_mean": mean_gap,
        "gap_filled_median": median_gap,
    }

    # ------------------------------
    # MQF plotting array
    # ------------------------------

    mqf_plot = np.full(
        mqf.shape,
        np.nan,
        dtype=np.float32,
    )

    mqf_land = mqf[land_mask].copy()

    mqf_land[
        ~np.isfinite(mqf_land)
    ] = 255

    mqf_land[
        mqf_land == 255
    ] = 3

    mqf_plot[
        land_mask
    ] = mqf_land

    del mqf_land

    # ------------------------------
    # Two-panel figure
    # ------------------------------

    fig = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=True,
        horizontal_spacing=0.08,
        subplot_titles=[
            "DNB_BRDF_Corrected_NTL",
            "Mandatory_Quality_Flag",
        ],
    )

    # Retrieve subplot domains so each colorbar remains
    # inside its respective panel.
    ntl_domain = fig.layout.xaxis.domain
    mqf_domain = fig.layout.xaxis2.domain

    ntl_colorbar_x = ntl_domain[1] + 0.05
    mqf_colorbar_x = mqf_domain[1] + 0.1

    # ------------------------------
    # DNB-BRDF panel
    # ------------------------------

    fig.add_trace(
        go.Heatmap(
            z=ntl,
            x=lons,
            y=lats,
            zmin=R14_DISPLAY_MIN,
            zmax=R14_DISPLAY_MAX,
            colorscale="Cividis",
            hoverinfo="skip",
            colorbar=dict(
                title=dict(
                    text="nW/cm²/sr",
                    side="top",
                    font=dict(size=12),
                ),
                tickvals=R14_NTL_TICKS,
                ticktext=[
                    f"{value:.1f}"
                    for value in R14_NTL_TICKS
                ],
                tickfont=dict(size=10),
                len=0.28,
                thickness=10,
                x=ntl_colorbar_x,
                xanchor="right",
                y=0.9,
                yanchor="middle",
                outlinewidth=0,
            ),
        ),
        row=1,
        col=1,
    )

    # ------------------------------
    # Mandatory Quality Flag panel
    # ------------------------------

    fig.add_trace(
        go.Heatmap(
            z=mqf_plot,
            x=lons,
            y=lats,
            zmin=-0.5,
            zmax=3.5,
            colorscale=R14_QF_COLORSCALE,
            hoverinfo="skip",
            colorbar=dict(
                title=dict(
                    text="Quality Flag",
                    side="top",
                    font=dict(size=12),
                ),
                tickvals=[0, 1, 2, 3],
                ticktext=R14_QF_LABELS,
                tickfont=dict(size=10),
                len=0.28,
                thickness=10,
                x=mqf_colorbar_x,
                xanchor="right",
                y=0.9,
                yanchor="middle",
                outlinewidth=0,
            ),
        ),
        row=1,
        col=2,
    )

    # ------------------------------
    # Regional boundary overlay
    # ------------------------------

    for column in [1, 2]:

        fig.add_trace(
            go.Scatter(
                x=R14_BOUNDARY_X,
                y=R14_BOUNDARY_Y,
                mode="lines",
                line=dict(
                    color="rgba(45,45,45,0.85)",
                    width=0.8,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

    # ------------------------------
    # Center subplot titles
    # ------------------------------

    for annotation in fig.layout.annotations:

        if annotation.text in {
            "DNB_BRDF_Corrected_NTL",
            "Mandatory_Quality_Flag",
        }:
            annotation.font = dict(
                family="Arial",
                size=11,
                color="black",
            )

            annotation.xanchor = "center"
            annotation.y = 1.025

    # ------------------------------
    # Statistics inside white boxes
    # ------------------------------

    fig.add_annotation(
        text=(
            f"DNB-BRDF Mean: {mean_ntl:.2f}; "
            f"Median: {median_ntl:.2f}<br>"
            f"Gap-filled Mean: {mean_gap:.2f}; "
            f"Median: {median_gap:.2f}"
        ),
        xref="x domain",
        yref="y domain",
        x=0.01,
        y=0.01,
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        align="left",
        font=dict(
            family="Arial",
            size=12,
            color="black",
        ),
        bgcolor="rgba(255,255,255,0.92)",
        bordercolor="rgba(80,80,80,0.85)",
        borderwidth=1,
        borderpad=2,
    )

    fig.add_annotation(
        text=(
            f"Spatial Completeness: "
            f"{spatial_completeness:.1f}%"
        ),
        xref="x2 domain",
        yref="y2 domain",
        x=0.01,
        y=0.01,
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        align="left",
        font=dict(
            family="Arial",
            size=12,
            color="black",
        ),
        bgcolor="rgba(255,255,255,0.92)",
        bordercolor="rgba(80,80,80,0.85)",
        borderwidth=1,
        borderpad=2,
    )

    # ------------------------------
    # Layout
    # ------------------------------

    fig.update_layout(
        template="plotly_white",
        width=600,
        height=400,
        margin=dict(
            l=45,
            r=25,
            t=45,
            b=35,
        ),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="#E5ECF6",
        font=dict(
            family="Arial",
            size=9,
            color="black",
        ),
        showlegend=False,
    )

    fig.update_xaxes(
        range=R14_LON_RANGE,
        tickformat=".1f",
        tickfont=dict(size=8),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.70)",
        gridwidth=1,
        zeroline=False,
        showline=False,
    )

    fig.update_yaxes(
        range=R14_LAT_RANGE,
        tickformat=".1f",
        tickfont=dict(size=8),
        showgrid=True,
        gridcolor="rgba(255,255,255,0.70)",
        gridwidth=1,
        zeroline=False,
        showline=False,
    )

    fig.update_yaxes(
        showticklabels=False,
        row=1,
        col=2,
    )

    del ntl
    del gap
    del mqf
    del mqf_plot
    del land_mask
    del valid_ntl_mask

    return fig, statistics


# ==========================================================
# Display all four Plotly figures in the notebook
# No dates are added to the figures.
# No files are exported by this block.
# ==========================================================

R14_PREVIEW_ROWS = []

for date_key, metadata in R14_DATES.items():

    preview_fig, preview_statistics = (
        build_r14_figure(date_key)
    )

    preview_fig.show(
        config={
            "displaylogo": False,
            "responsive": True,
        }
    )

    preview_statistics[
        "panel"
    ] = metadata["panel"]

    preview_statistics[
        "label"
    ] = metadata["label"]

    R14_PREVIEW_ROWS.append(
        preview_statistics
    )

    del preview_fig
    gc.collect()

R14_EVIDENCE = (
    pd.DataFrame(R14_PREVIEW_ROWS)
    .sort_values("date")
    .reset_index(drop=True)
)

display(R14_EVIDENCE)

In [ ]:
# R1.4 — Export revised Figure 3 panels
R14_EXPORT_DIR = PROJECT_ROOT / "outputs" / "revised_plots"
R14_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for date_key in R14_DATES:
    export_fig, _ = build_r14_figure(date_key)

    export_plotly_figure(
        export_fig,
        R14_EXPORT_DIR / f"fig_03_{date_key}",
        width=600,
        height=400,
        scale=3,
    )

    del export_fig
    gc.collect()

print(f"Saved Figure 3 panels to: {R14_EXPORT_DIR}")

## R1.7 — Figures 7 and 8 readability

> “In Figures 7 and 8, the lines are overly thick and heavily overlapping, which makes it difficult to distinguish the individual series and extract useful information.”

Figure 7 separates the eight Samar–Leyte GHSL classes into individual rows at τ=10%. Figure 8 shows selected configurations for Bohol (G1, τ=40%), Cebu (G1, τ=50%), Negros (G1, τ=15%) and Panay (G4, τ=5%). Native-daily observations (W=1) appear as faint points alongside 30-day trailing means (W=30), shown as smoothing sensitivity views. Full-record panels are paired with boxed −60/+180-day zooms around Haiyan and Rai, respectively. Larger text, clearer axes and compact labels report panel-specific r₁, r₃₀, N% and n. Zooms retain the full-record z-score normalisation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ----------------------------------------------------------
# Project and input paths
# ----------------------------------------------------------
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", Path.cwd()))
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "datasets").is_dir():
    PROJECT_ROOT = (
        Path.home()
        / "Library/CloudStorage/OneDrive-RMITUniversity"
        / "02 - CH1 - Reliability-Qualified Nighttime Lights (Technical Validation)"
        / "blackmarble-reliability-conditions"
    )

NTL_FOLDER = (
    PROJECT_ROOT / "datasets/vnp46a2/visayas_ghsl_masked_daily_stats"
)
NGCP_XLSX_PATH = (
    PROJECT_ROOT / "datasets/ngcp/NGCP_hourly_load.xlsx"
)

DATE_START = pd.Timestamp(globals().get("DATE_START", "2012-01-19"))
DATE_END = pd.Timestamp(globals().get("DATE_END", "2024-12-31"))
ANALYSIS_DAYS = (DATE_END - DATE_START).days + 1
VALUE_COL = globals().get("VALUE_COL", "Mean_by_sum")

# Preserve metadata/helpers already defined in the notebook.
if "GHSL_META" not in globals():
    GHSL_META = {
        "30": {"code": "G1", "label": "Urban Core", "color": "#ff0000"},
        "23-30": {"code": "G2", "label": "Dense Urban", "color": "#732600"},
        "22-30": {"code": "G3", "label": "Core & Inner", "color": "#a87000"},
        "21-30": {"code": "G4", "label": "Urban Fringe", "color": "#ffff00"},
        "13-30": {"code": "G5", "label": "Built-Up / Rural Mix", "color": "#375623"},
        "12-30": {"code": "G6", "label": "Low–Mid Rural", "color": "#abcd66"},
        "11-30": {"code": "G7", "label": "Rural Settlements", "color": "#cdf57a"},
        "10-30": {"code": "G8", "label": "All Land", "color": "#7ab6f5"},
    }

if "GHSL_ORDER" not in globals():
    GHSL_ORDER = [
        "30", "23-30", "22-30", "21-30",
        "13-30", "12-30", "11-30", "10-30",
    ]

if "STORM_MARKERS" not in globals():
    STORM_MARKERS = [
        ("Haiyan", "2013-11-08"),
        ("Rai", "2021-12-16"),
    ]

if "zscore" not in globals():
    def zscore(series):
        s = pd.Series(series).astype(float)
        sd = s.std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            return pd.Series(np.nan, index=s.index)
        return (s - s.mean()) / sd

# ----------------------------------------------------------
# Load regional GHSL-masked NTL
# ----------------------------------------------------------
region_names = {
    "CebuProvince": "Cebu",
    "SamarLeyte": "Samar-Leyte",
    "NegrosIsland": "Negros",
    "PanayIsland": "Panay",
    "BoholProvince": "Bohol",
}

csv_files = sorted(NTL_FOLDER.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No NTL CSV files found in: {NTL_FOLDER}")

frames = []

for file in csv_files:
    parts = file.name.replace("_stats.csv", "").split("_DNBBRDF_class")
    if len(parts) != 2:
        continue

    region, group = parts
    region = region_names.get(region, region)
    group = "30" if group == "30-only" else group

    try:
        frame = pd.read_csv(file)
    except Exception:
        frame = pd.read_csv(
            file, dtype=str, engine="python", on_bad_lines="skip"
        )

    frame["Region"] = region
    frame["Group"] = group
    frames.append(frame)

if not frames:
    raise RuntimeError("No matching regional NTL files were loaded.")

df_all = pd.concat(frames, ignore_index=True)
df_all["date"] = pd.to_datetime(
    df_all["date"], errors="coerce"
).dt.normalize()
df_all = df_all[
    df_all["date"].between(DATE_START, DATE_END)
].copy()

for column in [
    "Mean_by_sum", "NTL_max", "NTL_mean", "NTL_median",
    "NTL_min", "NTL_std", "Valid_px",
]:
    if column in df_all.columns:
        df_all[column] = pd.to_numeric(df_all[column], errors="coerce")

class_order = {group: i + 1 for i, group in enumerate(GHSL_ORDER)}
df_all["ClassID"] = df_all["Group"].map(class_order)
df_all = df_all.sort_values(
    ["Region", "ClassID", "date"]
).reset_index(drop=True)

# ----------------------------------------------------------
# Load NGCP 1 AM electricity load
# ----------------------------------------------------------
sheet_map = {
    "Panay": "PANAY HOURLY LOAD 2013-2024",
    "Negros": "NEGROS HOURLY LOAD 2013-2024",
    "Samar-Leyte": "LEY-SAM HOURLY LOAD 2013-2024",
    "Cebu": "CEBU HOURLY LOAD 2013-2024",
    "Bohol": "BOHOL LOAD 2013-2024",
}

if not NGCP_XLSX_PATH.is_file():
    raise FileNotFoundError(f"NGCP workbook not found: {NGCP_XLSX_PATH}")

load_series = {}

for region, sheet_name in sheet_map.items():
    frame = pd.read_excel(
        NGCP_XLSX_PATH, sheet_name=sheet_name, header=1
    )
    frame.columns = [str(c).strip() for c in frame.columns]
    frame = frame.rename(columns={frame.columns[0]: "DATE"})
    frame["DATE"] = pd.to_datetime(frame["DATE"], errors="coerce")
    frame = frame.dropna(subset=["DATE"]).set_index("DATE")

    hour_col = next(
        (
            c for c in ["1", "01", "1.0", "Hour 1", "HOUR 1", "1AM", "1am"]
            if c in frame.columns
        ),
        None,
    )
    if hour_col is None:
        numeric_columns = [
            c for c in frame.columns
            if str(c).replace(".0", "").isdigit()
        ]
        hour_col = (
            numeric_columns[0] if numeric_columns else frame.columns[0]
        )

    series = pd.to_numeric(frame[hour_col], errors="coerce")
    load_series[region] = series.loc[
        (series.index >= DATE_START) & (series.index <= DATE_END)
    ]

ngcp_df = pd.DataFrame(load_series).sort_index()

print(f"Loaded {len(df_all):,} NTL rows.")
print(f"NGCP regions: {', '.join(ngcp_df.columns)}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
def build_alignment_source(df_all, ngcp_df, region, tau):
    dates = pd.date_range(DATE_START, DATE_END, freq="D")

    ng = pd.to_numeric(ngcp_df[region], errors="coerce").copy()
    ng.index = pd.to_datetime(ng.index).normalize()
    if ng.index.has_duplicates:
        raise ValueError(f"Duplicate NGCP dates: {region}")
    ng = ng.sort_index().reindex(dates)

    # NGCP screening, applied once to the original input.
    window = ng.rolling(31, center=True, min_periods=7)
    median = window.median()
    mad = window.apply(
        lambda x: np.nanmedian(np.abs(x - np.nanmedian(x))),
        raw=True,
    )
    sigma = 1.4826022185 * mad
    flagged = (sigma > 0) & ((ng - median).abs() > 3 * sigma)
    ng = ng.mask(flagged)

    ng_daily = zscore(ng)
    ng_30d = ng_daily.rolling("30D", min_periods=1).mean()

    # Preserve the trace positions expected by your final layout cell.
    source = make_subplots(rows=2 * len(GHSL_ORDER), cols=2)
    all_values = [ng_daily.to_numpy(), ng_30d.to_numpy()]

    for i, group in enumerate(GHSL_ORDER):
        gdf = df_all[
            (df_all["Region"] == region)
            & (df_all["Group"] == group)
        ].copy()
        gdf = gdf[gdf["Valid_px"] > 0].copy()
        gdf["date"] = pd.to_datetime(gdf["date"]).dt.normalize()
        gdf = gdf.sort_values("date")

        if gdf.empty:
            raise ValueError(f"No NTL data: {region}, {group}")

        hi = np.nanpercentile(gdf[VALUE_COL], 95)
        gdf = gdf[gdf[VALUE_COL] <= hi].copy()

        if gdf.empty:
            raise ValueError(f"No NTL data after filtering: {group}")
        if gdf["date"].duplicated().any():
            raise ValueError(f"Duplicate NTL dates: {group}")

        gdf["SC_pct"] = gdf["Valid_px"] / gdf["Valid_px"].max() * 100
        gdf = gdf.set_index("date").reindex(dates)
        ntl = gdf[VALUE_COL].where(gdf["SC_pct"] > tau)

        paired = ntl.notna() & ng.notna()
        reference = ntl.loc[paired]
        sd = reference.std(ddof=0)

        if np.isfinite(sd) and sd > 0:
            ntl_scaled = (ntl - reference.mean()) / sd
        else:
            ntl_scaled = pd.Series(np.nan, index=dates)

        ntl_daily = ntl_scaled.where(paired)
        ntl_30d = ntl_scaled.rolling("30D", min_periods=1).mean()

        all_values.extend([
            ntl_daily.to_numpy(),
            ntl_30d.to_numpy(),
        ])

        # Order required by the final layout:
        # NGCP daily, NTL daily, NTL 30-day, NGCP 30-day.
        for series, mode, color in [
            (ng_daily, "markers", "#666666"),
            (ntl_daily, "markers", GHSL_META[group]["color"]),
            (ntl_30d, "lines", GHSL_META[group]["color"]),
            (ng_30d, "lines", "black"),
        ]:
            source.add_trace(
                go.Scatter(
                    x=dates.strftime("%Y-%m-%d").tolist(),
                    y=series.to_numpy(),
                    mode=mode,
                    marker=dict(color=color),
                    line=dict(color=color),
                    connectgaps=False,
                    showlegend=False,
                ),
                row=2 * i + 2,
                col=1,
            )

    values = np.concatenate(all_values)
    values = values[np.isfinite(values)]
    if not values.size:
        raise ValueError("No finite observations remain.")

    padding = max(0.04 * (values.max() - values.min()), 0.1)

    source.update_xaxes(
        type="date",
        range=[
            dates.min().strftime("%Y-%m-%d"),
            dates.max().strftime("%Y-%m-%d"),
        ],
        row=2, col=1,
    )
    source.update_yaxes(
        range=[values.min() - padding, values.max() + padding],
        row=2, col=1,
    )
    return source


REGION = "Samar-Leyte"
TAU = 25

fig_alignment_all8 = build_alignment_source(
    df_all,
    ngcp_df,
    region=REGION,
    tau=TAU,
)

In [ ]:
source = fig_alignment_all8
groups = list(GHSL_ORDER)

haiyan = pd.Timestamp("2013-11-08")
zoom_start = haiyan - pd.Timedelta(days=60)
zoom_end = haiyan + pd.Timedelta(days=180)

full_start, full_end = pd.to_datetime(source.layout.xaxis3.range)
views = [(full_start, full_end), (zoom_start, zoom_end)]

HEIGHT = 2100
TOP = 75
BOTTOM = 85
inner_height = HEIGHT - TOP - BOTTOM

fig = make_subplots(
    rows=len(groups),
    cols=2,
    shared_xaxes="columns",
    column_widths=[0.75, 0.25],
    horizontal_spacing=0.045,
    vertical_spacing=0.018,
)

old_limits = source.layout.yaxis3.range
y_limits = [
    min(float(old_limits[0]), -2.2),
    max(float(old_limits[1]), 2.2),
]


def correlation_and_count(ntl_trace, ngcp_trace):
    ntl = pd.Series(
        np.asarray(ntl_trace.y, dtype=float),
        index=pd.to_datetime(ntl_trace.x),
        name="NTL",
    )
    load = pd.Series(
        np.asarray(ngcp_trace.y, dtype=float),
        index=pd.to_datetime(ngcp_trace.x),
        name="NGCP",
    )
    paired = pd.concat([ntl, load], axis=1).dropna()
    r = paired["NTL"].corr(paired["NGCP"]) if len(paired) >= 2 else np.nan
    return r, len(paired)


for i, group in enumerate(groups):
    row = i + 1
    meta = GHSL_META[group]

    # Always take the full-record data, including dates beyond
    # the previous +120-day zoom.
    original_traces = [
        t for t in source.data
        if t.type == "scatter"
        and t.xaxis == f"x{4 * i + 3}"
        and not t.showlegend
    ]

    base_bottom, base_top = fig.get_subplot(row, 1).yaxis.domain
    plot_top = base_top - 38 / inner_height

    for col, (start, end) in enumerate(views, start=1):
        panel_traces = []

        for j, trace in enumerate(original_traces):
            dates = pd.DatetimeIndex(pd.to_datetime(trace.x))
            selected = (dates >= start) & (dates <= end)

            copied = go.Scatter(trace.to_plotly_json())
            copied.x = dates[selected].strftime("%Y-%m-%d").tolist()
            copied.y = np.asarray(trace.y, dtype=float)[selected]
            copied.showlegend = False

            is_ngcp = j in (0, 3)

            if copied.mode == "markers":
                copied.marker.size = 3.2 if col == 1 else 4.8
                copied.marker.opacity = (
                    (0.13 if col == 1 else 0.22)
                    if is_ngcp else
                    (0.24 if col == 1 else 0.38)
                )
            else:
                copied.line.width = (
                    (1.9 if col == 1 else 2.5)
                    if is_ngcp else
                    (1.7 if col == 1 else 2.2)
                )
                copied.connectgaps = False

            panel_traces.append(copied)
            fig.add_trace(copied, row=row, col=col)

        # Trace order: NGCP daily, NTL daily, NTL mean, NGCP mean.
        ng_daily, ntl_daily, ntl_30d, ng_30d = panel_traces
        r1, count = correlation_and_count(ntl_daily, ng_daily)
        r30, _ = correlation_and_count(ntl_30d, ng_30d)

        possible_days = (
            ANALYSIS_DAYS if col == 1 else (end - start).days + 1
        )
        coverage = 100 * count / possible_days

        r1_text = f"{r1:.2f}" if np.isfinite(r1) else "—"
        r30_text = f"{r30:.2f}" if np.isfinite(r30) else "—"

        label = (
            f"{meta['code']}: {meta['label']}"
            if col == 1 else meta["code"]
        )
        dot = f"<span style='color:{meta['color']}'>●</span>"
        line = f"<span style='color:{meta['color']}'>━</span>"

        # One text object, no title and no line breaks.
        key = (
            f"{label} {dot} r₁={r1_text}  "
            f"{line} r₃₀={r30_text}  "
            f"N={coverage:.0f}%  n={count}"
        )

        x_domain = fig.get_subplot(row, col).xaxis.domain
        fig.add_annotation(
            x=x_domain[1],
            y=base_top,
            xref="paper",
            yref="paper",
            text=key,
            xanchor="right",
            yanchor="top",
            showarrow=False,
            font=dict(
                family="Arial",
                size=18 if col == 1 else 17,
                color="#333333",
            ),
        )

        fig.update_yaxes(
            domain=[base_bottom, plot_top],
            range=y_limits,
            tickmode="array",
            tickvals=[-2, 0, 2],
            ticktext=["−2", "0", "+2"],
            tickfont=dict(size=17),
            showticklabels=True,
            zeroline=True,
            zerolinecolor="#dddddd",
            row=row, col=col,
        )
        fig.update_xaxes(
            type="date",
            range=[
                start.strftime("%Y-%m-%d"),
                end.strftime("%Y-%m-%d"),
            ],
            showticklabels=False,
            showgrid=False,
            row=row, col=col,
        )

        if col == 1:
            fig.update_yaxes(
                title_text="z-score",
                title_font=dict(size=19),
                row=row, col=col,
            )
            fig.add_vrect(
                x0=zoom_start.strftime("%Y-%m-%d"),
                x1=zoom_end.strftime("%Y-%m-%d"),
                line_color="#555555",
                line_width=1.5,
                fillcolor="rgba(0,0,0,0)",
                layer="above",
                row=row, col=col,
            )
        else:
            for update_axis in (fig.update_xaxes, fig.update_yaxes):
                update_axis(
                    showline=True,
                    mirror=True,
                    linecolor="#555555",
                    linewidth=1.5,
                    row=row, col=col,
                )

        for _, event_date in STORM_MARKERS:
            if start <= pd.Timestamp(event_date) <= end:
                fig.add_vline(
                    x=event_date,
                    line_color="black",
                    line_dash="dash",
                    line_width=0.9,
                    opacity=0.5,
                    row=row, col=col,
                )

# Shared NGCP key: one text object guarantees one horizontal line.
fig.add_annotation(
    x=0.6,
    y=0.92,
    xref="paper",
    yref="paper",
    text=(
        f"NGCP Electric Load<br>"
        "<span style='color:#666666'>●</span> W=1 "
        "<span style='color:black'>━</span> W=30"
    ),
    xanchor="left",
    yanchor="top",
    align="left",
    showarrow=False,
    font=dict(family="Arial", size=16, color="#333333"),
)
fig.add_annotation(
    x=0.72,
    y=0.975,
    xref="paper",
    yref="paper",
    text=f"(τ = {TAU}%)",
    xanchor="right",
    yanchor="bottom",
    showarrow=False,
    font=dict(size=19),
)

fig.update_xaxes(
    showticklabels=True,
    title_text="Date",
    title_font=dict(size=19),
    tickformat="%Y",
    dtick="M48",
    tickfont=dict(size=17),
    tickangle=0,
    row=len(groups), col=1,
)
fig.update_xaxes(
    showticklabels=True,
    title_text="Date",
    title_font=dict(size=19),
    tickformat="%b %Y",
    dtick="M3",
    tickfont=dict(size=17),
    tickangle=0,
    row=len(groups), col=2,
)

fig.update_layout(
    template="plotly_white",
    width=1550,
    height=HEIGHT,
    margin=dict(t=TOP, b=BOTTOM, l=95, r=45),
    font=dict(family="Arial", size=18, color="#333333"),
    paper_bgcolor="rgba(0,0,0,0)",
    showlegend=False,
)

fig_alignment_paper_final = fig
fig_alignment_paper_final.show()

In [ ]:
SETTINGS = [
    ("Bohol",  "30",    40, (-3, 3)),
    ("Cebu",   "30",    50, (-3, 3)),
    ("Negros", "30",    15, None),
    ("Panay",  "21-30",  5, None),
]

dates = pd.date_range(DATE_START, DATE_END, freq="D")
event = pd.Timestamp("2021-12-16")
zoom_start = event - pd.Timedelta(days=60)
zoom_end = event + pd.Timedelta(days=180)

ngcp_outlier_tables = {}


def prepare_selected_setting(region, group, tau):
    # Screen each region's NGCP series once.
    ng = pd.to_numeric(ngcp_df[region], errors="coerce").copy()
    ng.index = pd.to_datetime(ng.index).normalize()
    if ng.index.has_duplicates:
        raise ValueError(f"Duplicate NGCP dates: {region}")
    ng = ng.sort_index().reindex(dates)

    window = ng.rolling(31, center=True, min_periods=7)
    median = window.median()
    mad = window.apply(
        lambda x: np.nanmedian(np.abs(x - np.nanmedian(x))),
        raw=True,
    )
    sigma = 1.4826022185 * mad
    flagged = (sigma > 0) & ((ng - median).abs() > 3 * sigma)

    ngcp_outlier_tables[region] = pd.DataFrame({
        "NGCP_MW": ng,
        "Lower_limit_MW": median - 3 * sigma,
        "Upper_limit_MW": median + 3 * sigma,
    }).loc[flagged]
    ng = ng.mask(flagged)

    # Select only the columns needed for this configuration.
    gdf = df_all.loc[
        df_all["Region"].eq(region)
        & df_all["Group"].eq(group)
        & df_all["Valid_px"].gt(0),
        ["date", VALUE_COL, "Valid_px"],
    ].copy()

    if gdf.empty:
        raise ValueError(f"No NTL observations: {region}, {group}")

    hi = np.nanpercentile(gdf[VALUE_COL], 95)
    gdf = gdf.loc[gdf[VALUE_COL].le(hi)].copy()
    if gdf.empty:
        raise ValueError(f"No NTL observations after filtering: {region}")
    if gdf["date"].duplicated().any():
        raise ValueError(f"Duplicate NTL dates: {region}, {group}")

    gdf["SC_pct"] = gdf["Valid_px"] / gdf["Valid_px"].max() * 100
    gdf = gdf.set_index("date").sort_index().reindex(dates)
    ntl = gdf[VALUE_COL].where(gdf["SC_pct"] > tau)

    paired = ntl.notna() & ng.notna()
    reference = ntl.loc[paired]
    sd = reference.std(ddof=0)

    ntl_scaled = (
        (ntl - reference.mean()) / sd
        if np.isfinite(sd) and sd > 0
        else pd.Series(np.nan, index=dates)
    )
    ng_daily = zscore(ng)

    # Smoothed curves are daily estimates from available observations.
    # Windows with no observations remain missing.
    return pd.DataFrame({
        "ntl1": ntl_scaled.where(paired),
        "ntl30": ntl_scaled.rolling("30D", min_periods=1).mean(),
        "ng1": ng_daily,
        "ng30": ng_daily.rolling("30D", min_periods=1).mean(),
    })


# Prepare each configuration once; reuse it for both columns.
prepared = {
    (region, group, tau): prepare_selected_setting(region, group, tau)
    for region, group, tau, _ in SETTINGS
}

HEIGHT, TOP, BOTTOM = 1250, 75, 85
inner_height = HEIGHT - TOP - BOTTOM

fig = make_subplots(
    rows=len(SETTINGS),
    cols=2,
    shared_xaxes="columns",
    column_widths=[0.75, 0.25],
    horizontal_spacing=0.045,
    vertical_spacing=0.035,
)

views = [
    (pd.Timestamp(DATE_START), pd.Timestamp(DATE_END)),
    (zoom_start, zoom_end),
]

for row, (region, group, tau, fixed_limits) in enumerate(SETTINGS, 1):
    data = prepared[(region, group, tau)]
    meta = GHSL_META[group]

    if fixed_limits is not None:
        limits = list(fixed_limits)
        ticks = [-3, 0, 3]
    else:
        values = data.to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if not values.size:
            raise ValueError(f"No finite comparison data: {region}")
        padding = max(0.04 * np.ptp(values), 0.1)
        limits = [
            min(values.min() - padding, -2.2),
            max(values.max() + padding, 2.2),
        ]
        ticks = [-2, 0, 2]

    base_bottom, base_top = fig.get_subplot(row, 1).yaxis.domain
    plot_top = base_top - 38 / inner_height

    for col, (start, end) in enumerate(views, 1):
        view = data.loc[start:end]
        x = view.index.strftime("%Y-%m-%d").tolist()

        paired = view[["ntl1", "ng1"]].notna().all(axis=1)
        count = int(paired.sum())
        possible_days = (
            ANALYSIS_DAYS if col == 1 else (end - start).days + 1
        )
        coverage = 100 * count / possible_days

        if count >= 2:
            r1 = view.loc[paired, "ntl1"].corr(
                view.loc[paired, "ng1"]
            )
            r30 = view.loc[paired, "ntl30"].corr(
                view.loc[paired, "ng30"]
            )
        else:
            r1 = r30 = np.nan

        r1_text = f"{r1:.2f}" if np.isfinite(r1) else "—"
        r30_text = f"{r30:.2f}" if np.isfinite(r30) else "—"

        for key, is_ngcp, mode in [
            ("ng1", True, "markers"),
            ("ntl1", False, "markers"),
            ("ntl30", False, "lines"),
            ("ng30", True, "lines"),
        ]:
            color = (
                ("#666666" if mode == "markers" else "black")
                if is_ngcp else meta["color"]
            )

            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=view[key].to_numpy(),
                    mode=mode,
                    marker=dict(
                        color=color,
                        size=3.2 if col == 1 else 4.8,
                        opacity=(
                            (0.13 if col == 1 else 0.22)
                            if is_ngcp else
                            (0.24 if col == 1 else 0.38)
                        ),
                    ),
                    line=dict(
                        color=color,
                        width=(
                            (1.9 if col == 1 else 2.5)
                            if is_ngcp else
                            (1.7 if col == 1 else 2.2)
                        ),
                    ),
                    connectgaps=False,
                    showlegend=False,
                    hovertemplate="%{x}<br>z=%{y:.2f}<extra></extra>",
                ),
                row=row, col=col,
            )

        label = (
            f"{region} · {meta['code']}: {meta['label']} · τ={tau}%"
            if col == 1 else meta["code"]
        )
        dot = f"<span style='color:{meta['color']}'>●</span>"
        line = f"<span style='color:{meta['color']}'>━</span>"

        fig.add_annotation(
            x=fig.get_subplot(row, col).xaxis.domain[1],
            y=base_top,
            xref="paper",
            yref="paper",
            text=(
                f"{label}  {dot} r₁={r1_text}  "
                f"{line} r₃₀={r30_text}  "
                f"N={coverage:.0f}%  n={count}"
            ),
            xanchor="right",
            yanchor="top",
            showarrow=False,
            font=dict(size=18 if col == 1 else 17),
        )

        fig.update_yaxes(
            domain=[base_bottom, plot_top],
            range=limits,
            tickmode="array",
            tickvals=ticks,
            ticktext=[
                f"−{abs(v)}" if v < 0 else f"+{v}" if v > 0 else "0"
                for v in ticks
            ],
            tickfont=dict(size=17),
            showticklabels=True,
            zeroline=True,
            zerolinecolor="#dddddd",
            row=row, col=col,
        )
        fig.update_xaxes(
            type="date",
            range=[
                start.strftime("%Y-%m-%d"),
                end.strftime("%Y-%m-%d"),
            ],
            showticklabels=False,
            showgrid=False,
            row=row, col=col,
        )

        if col == 1:
            fig.update_yaxes(
                title_text="z-score",
                title_font=dict(size=19),
                row=row, col=col,
            )
            fig.add_vrect(
                x0=zoom_start.strftime("%Y-%m-%d"),
                x1=zoom_end.strftime("%Y-%m-%d"),
                line_color="#555555",
                line_width=1.5,
                fillcolor="rgba(0,0,0,0)",
                layer="above",
                row=row, col=col,
            )
        else:
            for update_axis in (fig.update_xaxes, fig.update_yaxes):
                update_axis(
                    showline=True,
                    mirror=True,
                    linecolor="#555555",
                    linewidth=1.5,
                    row=row, col=col,
                )

        for _, event_date in STORM_MARKERS:
            if start <= pd.Timestamp(event_date) <= end:
                fig.add_vline(
                    x=event_date,
                    line_color="black",
                    line_dash="dash",
                    line_width=0.9,
                    opacity=0.5,
                    row=row, col=col,
                )

# Preserve your refined shared-key placement.
fig.add_annotation(
    x=0.6,
    y=0.82,
    xref="paper",
    yref="paper",
    text=(
        "NGCP Electric Load<br>"
        "<span style='color:#666666'>●</span> W=1 "
        "<span style='color:black'>━</span> W=30"
    ),
    xanchor="left",
    yanchor="top",
    align="left",
    showarrow=False,
    font=dict(family="Arial", size=16, color="#333333"),
)

for col, tickformat, dtick in [
    (1, "%Y", "M48"),
    (2, "%b %Y", "M3"),
]:
    fig.update_xaxes(
        showticklabels=True,
        title_text="Date",
        title_font=dict(size=19),
        tickformat=tickformat,
        dtick=dtick,
        tickfont=dict(size=17),
        tickangle=0,
        row=len(SETTINGS), col=col,
    )

fig.update_layout(
    template="plotly_white",
    width=1550,
    height=HEIGHT,
    margin=dict(t=TOP, b=BOTTOM, l=95, r=45),
    font=dict(family="Arial", size=18, color="#333333"),
    paper_bgcolor="rgba(0,0,0,0)",
    showlegend=False,
)

fig_selected_regions_rai = fig
fig_selected_regions_rai.show()

In [ ]:
REVISION_DIR = PROJECT_ROOT / "outputs" / "revised_plots"

for figure, filename in [
    (fig_alignment_paper_final, "fig_07_samar_leyte"),
    (fig_selected_regions_rai, "fig_08_selected_regions_rai"),
]:
    export_plotly_figure(
        figure,
        REVISION_DIR / filename,
        width=figure.layout.width,
        height=figure.layout.height,
        scale=3,
    )